# Relative vs absolute quality signal

Section 5.7 found the quality term fails when clean clients differ in local
loss more than corruption moves it. `relative_quality` scores each client
against its own first-round value to cancel the per-client difficulty term.

This runs both variants at 10 seeds on **both** corpora, evaluated on the
**validation** split — selection never touches test.


In [ ]:
import glob, os, shutil, sys, zipfile
REPO='/kaggle/working/repo'
hits=glob.glob('/kaggle/input/**/paper/build_paper.py', recursive=True)
assert hits, 'attach the aha-fed-code dataset'
src=os.path.dirname(os.path.dirname(hits[0]))
if not os.path.isdir(os.path.join(REPO,'paper')):
    shutil.copytree(src, REPO, dirs_exist_ok=True)
os.chdir(REPO); sys.path.insert(0, REPO)
print('staged:', os.getcwd())


In [ ]:
!pip install -q medmnist opacus


In [ ]:
import subprocess, sys
PROBE=('import torch;'
       'a=torch.randn(4,3,8,8,device="cuda");'
       'w=torch.randn(2,3,3,3,device="cuda");'
       'torch.nn.functional.conv2d(a,w).sum().item();'
       'print(torch.cuda.get_device_name(0), torch.__version__)')
r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
print(r.stdout.strip() or r.stderr.strip()[-400:])
if r.returncode!=0:
    print('installing cu118 build for this GPU ...')
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'torch==2.4.1','torchvision==0.19.1',
                    '--index-url','https://download.pytorch.org/whl/cu118'],check=True)
    r=subprocess.run([sys.executable,'-c',PROBE],capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr.strip()[-400:])
    if r.returncode!=0: raise SystemExit('no working torch/GPU combination')


In [ ]:
from paper.run_real_experiments import run_one, OUT_DIR

SEEDS=[42,123,456,7,13,21,99,2024,31337,555]
PNEU={'dataset':'pneumoniamnist','num_classes':2,'samples_per_client':300}
PATH={'dataset':'pathmnist','num_classes':9,'samples_per_client':1000}

for s in SEEDS:
    for rel,tag in [(True,'relq'),(False,'absq')]:
        run_one(OUT_DIR/f'pneuval_{tag}_s{s}.json', aggregator='adaptive',
                alpha=1.0, beta=4.0, gamma=4.0, relative_quality=rel,
                seed=s, num_rounds=20, eval_split='val', **PNEU)
print('PneumoniaMNIST done')


In [ ]:
for s in SEEDS:
    for rel,tag in [(True,'relq'),(False,'absq')]:
        run_one(OUT_DIR/f'pathval_{tag}_s{s}.json', aggregator='adaptive',
                alpha=1.0, beta=4.0, gamma=4.0, relative_quality=rel,
                seed=s, eval_split='val', **PATH)
print('PathMNIST done')


In [ ]:
import glob, json, statistics as st
from pathlib import Path
from scipy.stats import wilcoxon

def fin(f):
    d=json.loads(Path(f).read_text()); return d.get('final') or {}

for corpus in ['pneuval','pathval']:
    rel={Path(f).stem.split('_s')[-1]: fin(f)
         for f in sorted(glob.glob(f'runs/paper/real/{corpus}_relq_s*.json'))}
    ab={Path(f).stem.split('_s')[-1]: fin(f)
        for f in sorted(glob.glob(f'runs/paper/real/{corpus}_absq_s*.json'))}
    common=sorted(set(rel)&set(ab))
    if not common: continue
    print(f'=== {corpus} ({len(common)} paired seeds) ===')
    for metric in ['ece','accuracy','auroc']:
        r=[rel[c][metric] for c in common]; a=[ab[c][metric] for c in common]
        p=wilcoxon(r,a).pvalue if len(common)>=6 else float('nan')
        better='relative' if (st.mean(r)<st.mean(a))==(metric=='ece') else 'absolute'
        print(f'  {metric:9s} rel {st.mean(r):.4f}  abs {st.mean(a):.4f}  '
              f'p={p:.4f}  -> {better} better')
    print()


In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/relq_results.zip runs/paper/real
!ls -lh /kaggle/working/relq_results.zip
